In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

Ahora hay bases por mes, entonces calculamos para cada mes

In [2]:
data1 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/BBDD_PUBLICACION_ ENE 21_SPSS/enemdu_persona_2021_01.sav", convert_categoricals=False) # para bases de stata
data2 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/BBDD_PUBLICACION_ FEB 21_SPSS/BBDD_PUBLICACION_ FEB  21_SPSS/enemdu_persona_2021_02.sav", convert_categoricals=False) # para bases de stata
data3 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/BBDD_PUBLICACION_ MAR 21_SPSS/enemdu_persona_2021_03.sav", convert_categoricals=False) # para bases de stata

data4 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/BBDD_PUBLICACION_ABRIL 21_SPSS/enemdu_persona_2021_04.sav", convert_categoricals=False) # para bases de stata
data5 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/BBDD_PUBLICACION_MAYO 21_SPSS/enemdu_persona_2021_05.sav", convert_categoricals=False) # para bases de stata
data6 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_06_SPSS/enemdu_persona_2021_06.sav", convert_categoricals=False) # para bases de stata

data7 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_07_SPSS/enemdu_persona_2021_07.sav", convert_categoricals=False) # para bases de stata
data8 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_08_SPSS/enemdu_persona_2021_08.sav", convert_categoricals=False) # para bases de stata
data9 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_09_SPSS/enemdu_persona_2021_09.sav", convert_categoricals=False) # para bases de stata

data10 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_10_SPSS/enemdu_persona_2021_10.sav", convert_categoricals=False) # para bases de stata
data11 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_11_SPSS/enemdu_persona_2021_11.sav", convert_categoricals=False) # para bases de stata
data12 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2021/raw/1_BDD_ENEMDU_2021_12_SPSS/enemdu_persona_2021_12.sav", convert_categoricals=False) # para bases de stata

In [3]:
pd.Series(data1.columns).to_csv("col.csv")

## Revisar los datos

| enero   | febrero | marzo   | abril   | mayo    | junio   | julio   | agosto  | septiembre | octubre | noviembre | diciembre |
|---------|---------|---------|---------|---------|---------|---------|---------|------------|---------|-----------|-----------|
| area| area| area| area| area| area| area| area| area| area| area| area|
| upm | upm | upm | upm | upm | ciudad| ciudad| ciudad| ciudad| ciudad| ciudad| ciudad|
| | | | | | vivienda| vivienda| vivienda| vivienda| vivienda| vivienda| vivienda|
| | | | | | hogar| hogar| hogar| hogar| hogar| hogar| hogar|
| p02| p02| p02| p02| p02| p02| p02| p02| p02| p02| p02| p02|
| p03| p03| p03| p03| p03| p03| p03| p03| p03| p03| p03| p03|
| p66| p66| p66| p66| p66| p66| p66| p66| p66| p66| p66| p66|
| fexp| fexp| fexp| fexp| fexp| fexp| fexp| fexp| fexp| fexp| fexp| fexp|
| p20| p20| p20| p20| p20| p20| p20| p20| p20| p20| p20| p20|
| | | | | | id_hogar| id_hogar| id_hogar| id_hogar| id_hogar| id_hogar| id_hogar|
| upm| upm| upm| upm| upm| | | | | | | |
| p01| p01| p01| p01| p01| | | | | | | |
| p04| p04| p04| p04| p04| | | | | | | |
| estrato| estrato| estrato| estrato| estrato| | | | | | | |

En esta encuesta tenemos una base para cada mes, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables p66 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

En los meses desde enero mayo no está disponible la variable de ciudad así que usaremos los primeros dígitos del upm para estos meses que equivalen a los geocódigos de provincia.

Elegimos hacer los cálculos en las bases mensuales para poder usar la representatividad mensual, con un promedio simple debería bastar para tener el resultado del trimestre.

Hay un problmea con algunos meses que no tienen de forma explicita el id de hogar por lo qu elo vamos a construir, con el numero de persona p01 y el parentezco del hogar p04=2 es el jefe de hogar 8453 en marzo, segun el INEC, el id de hogar equivale a concatenera ciudad, conglomerado, panel, vivienda, hogar y upm equivale a ciudad, conglomerado, panel no es necesario porque consideramos solo un mes y hogar lo sacaremos de p01 y p04.

Ahora usamos p20 que pregunta si trabajo la semana pasada, si se cumple vamos a asumir que trabajo durante todo el mes, ahora en adelante capturaremos incluso de mejor forma el mercado laboral, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [4]:
columnas = pd.Index(['area', 'ciudad', 'vivienda', 
                     'hogar', 'p66', 'p01', 'p04',
            'fexp', 'p02', 'p03', 'p20', 'id_hogar',
            'upm', 'estrato'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data1 = data1[columnas.intersection(data1.columns)]
data2 = data2[columnas.intersection(data2.columns)]
data3 = data3[columnas.intersection(data3.columns)]
data4 = data4[columnas.intersection(data4.columns)]
data5 = data5[columnas.intersection(data5.columns)]
data6 = data6[columnas.intersection(data6.columns)]
data7 = data7[columnas.intersection(data7.columns)]
data8 = data8[columnas.intersection(data8.columns)]
data9 = data9[columnas.intersection(data9.columns)]
data10 = data10[columnas.intersection(data10.columns)]
data11 = data11[columnas.intersection(data11.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [6]:
data12['p66'].value_counts().sort_index(ascending=False)

p66
999999.0     46
12000.0       1
6800.0        1
5500.0        1
5000.0        3
           ... 
15.0          2
12.0          1
10.0          3
8.0           2
0.0         193
Name: count, Length: 500, dtype: int64

In [7]:
data1['p66'] = pd.to_numeric(data1['p66'], errors='coerce')
data1['p66'] = data1['p66'].apply(lambda x: np.nan if x > 11000 else x)
data1['p66'] = data1['p66'].apply(lambda x: np.nan if x < 0 else x)

data2['p66'] = pd.to_numeric(data2['p66'], errors='coerce')
data2['p66'] = data2['p66'].apply(lambda x: np.nan if x > 15000 else x)
data2['p66'] = data2['p66'].apply(lambda x: np.nan if x < 0 else x)

data3['p66'] = pd.to_numeric(data3['p66'], errors='coerce')
data3['p66'] = data3['p66'].apply(lambda x: np.nan if x > 10000 else x)
data3['p66'] = data3['p66'].apply(lambda x: np.nan if x < 0 else x)

data4['p66'] = pd.to_numeric(data4['p66'], errors='coerce')
data4['p66'] = data4['p66'].apply(lambda x: np.nan if x > 7000 else x)
data4['p66'] = data4['p66'].apply(lambda x: np.nan if x < 0 else x)

data5['p66'] = pd.to_numeric(data5['p66'], errors='coerce')
data5['p66'] = data5['p66'].apply(lambda x: np.nan if x > 5000 else x)
data5['p66'] = data5['p66'].apply(lambda x: np.nan if x < 0 else x)

data6['p66'] = pd.to_numeric(data6['p66'], errors='coerce')
data6['p66'] = data6['p66'].apply(lambda x: np.nan if x > 10000 else x)
data6['p66'] = data6['p66'].apply(lambda x: np.nan if x < 0 else x)

data7['p66'] = pd.to_numeric(data7['p66'], errors='coerce')
data7['p66'] = data7['p66'].apply(lambda x: np.nan if x > 7000 else x)
data7['p66'] = data7['p66'].apply(lambda x: np.nan if x < 0 else x)

data8['p66'] = pd.to_numeric(data8['p66'], errors='coerce')
data8['p66'] = data8['p66'].apply(lambda x: np.nan if x > 9000 else x)
data8['p66'] = data8['p66'].apply(lambda x: np.nan if x < 0 else x)

data9['p66'] = pd.to_numeric(data9['p66'], errors='coerce')
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x > 6900 else x)
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x < 0 else x)

data10['p66'] = pd.to_numeric(data10['p66'], errors='coerce')
data10['p66'] = data10['p66'].apply(lambda x: np.nan if x > 7500 else x)
data10['p66'] = data10['p66'].apply(lambda x: np.nan if x < 0 else x)

data11['p66'] = pd.to_numeric(data11['p66'], errors='coerce')
data11['p66'] = data11['p66'].apply(lambda x: np.nan if x > 8000 else x)
data11['p66'] = data11['p66'].apply(lambda x: np.nan if x < 0 else x)

data12['p66'] = pd.to_numeric(data12['p66'], errors='coerce')
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x > 12000 else x)
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x < 0 else x)

In [8]:
for base in [data1, data2, data3, data4, data5, data6, 
             data7, data8, data9, data10, data11, data12]:
    base['ingr'] = base['p66']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [9]:
for base in [data1, data2, data3, data4, data5, data6, 
             data7, data8, data9, data10, data11, data12]:
    base['ingr'] = base.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [10]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2021]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [11]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [12]:
data1['ciudad'] = data1['upm'].apply(lambda x: x[:5])
data2['ciudad'] = data2['upm'].apply(lambda x: x[:5])
data3['ciudad'] = data3['upm'].apply(lambda x: x[:5])
data4['ciudad'] = data4['upm'].apply(lambda x: x[:5])
data5['ciudad'] = data5['upm'].apply(lambda x: x[:5])

In [13]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    print(base['ciudad'][0])
    fac = base['ciudad'].apply(lambda x: len(str(x))).min()
    print(fac)

01015
5
01015
5
01015
5
01015
5
01015
5
10150.0
7
10150.0
7
10150.0
7
10150.0
7
10150.0
7
10150.0
7
10150.0
7


In [98]:
# Corregimos los códigos para usarlos cómo texto
for base in [data1, data2, data3, data4, data5]:
    base['ciudad'] = base['ciudad'].apply(str)
    base['ciudad'] = base['ciudad'].apply(lambda x: '0' + x if len(x) == 4 else x)
    base['ciudad_2'] = base['ciudad'].apply(lambda x: x[:4])

for base in [data6, data7, data8, data9, data10, data11,
             data12]:
    base['ciudad'] = base['ciudad'].apply(str)
    base['ciudad'] = base['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)
    base['ciudad_2'] = base['ciudad'].apply(lambda x: x[:4])

Diccionario ciudades disponibles

In [99]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

for base in [data6, data7, data8, data9,
             data10, data11, data12]:
    base['ciudad_asignada'] = base['ciudad_2'].apply(get_parroquia)

for base in [data1, data2, data3, data4,
             data5]:
    base['ciudad_asignada'] = base['ciudad_2'].apply(get_parroquia)

In [100]:
data8['ciudad_asignada'].value_counts()

ciudad_asignada
Sierra        5685
Costa         5607
Quito         3800
Nacional      3725
Guayaquil     3133
Machala       2490
Cuenca        2278
Ambato        2037
Esmeraldas     742
Loja           508
Manta           87
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [101]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [102]:
data1['ipc'] = data1.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data1['ipc_base'] = data1.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data2['ipc'] = data2.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data2['ipc_base'] = data2.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data3['ipc'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data3['ipc_base'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data4['ipc'] = data4.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data4['ipc_base'] = data4.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data5['ipc'] = data5.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data5['ipc_base'] = data5.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data6['ipc'] = data6.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data6['ipc_base'] = data6.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data7['ipc'] = data7.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data7['ipc_base'] = data7.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data8['ipc'] = data8.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data8['ipc_base'] = data8.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data9['ipc'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data10['ipc'] = data10.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data10['ipc_base'] = data10.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

data11['ipc'] = data11.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data11['ipc_base'] = data11.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

data12['ipc'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [103]:
# Calculamos el deflactor
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base['def'] = (base['ipc_base'] / base['ipc'])

In [104]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base['ingr_r'] = base['ingr'] * base['def']

In [105]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    print(base['ingr_r'].mean())

688.1725838302522
446.36115511141674
440.3543389776097
467.432415960233
435.7250763421248
453.28804721061283
440.36334690776476
433.69146888705257
514.1412994083227
442.1068251244767
433.9745671891735
455.3174809269609


## Regiones

In [114]:
# Corregimos los códigos para usarlos cómo texto
for base in [data1, data2, data3, data4, data5]:
    base['ciudad'] = base['ciudad'].apply(str)
    base['ciudad'] = base['ciudad'].apply(lambda x: '0' + x if len(x) == 4 else x)
    base['ciudad_2'] = base['ciudad'].apply(lambda x: x[:2])

for base in [data6, data7, data8, data9, data10, data11,
             data12]:
    base['ciudad'] = base['ciudad'].apply(str)
    base['ciudad'] = base['ciudad'].apply(lambda x: '0' + x if len(x) == 7 else x)
    base['ciudad_2'] = base['ciudad'].apply(lambda x: x[:2])

In [115]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [116]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base['region'] = base['ciudad_2'].map(codigo_region)

In [118]:
data1['region'].value_counts()

region
Sierra                  7516
Pichincha               3929
Guayas                  3904
Costa, Santo Domingo    3799
Amazonía                3333
El Oro                  3238
Azuay                   2583
Manabí                   664
Los Ríos                 545
Galápagos                293
Name: count, dtype: int64

In [119]:
data2['region'].value_counts()

region
Sierra                  7442
Pichincha               4037
Guayas                  3810
Costa, Santo Domingo    3591
Amazonía                3422
El Oro                  3160
Azuay                   2586
Manabí                   673
Los Ríos                 535
Galápagos                267
Name: count, dtype: int64

In [120]:
data3['region'].value_counts()

region
Sierra                  7510
Pichincha               3881
Guayas                  3819
Costa, Santo Domingo    3679
Amazonía                3321
El Oro                  3167
Azuay                   2619
Manabí                   717
Los Ríos                 535
Galápagos                254
Name: count, dtype: int64

In [121]:
data4['region'].value_counts()

region
Sierra                  7695
Pichincha               4197
Guayas                  3840
Costa, Santo Domingo    3830
Amazonía                3430
El Oro                  3341
Azuay                   2739
Manabí                   665
Los Ríos                 546
Galápagos                131
Name: count, dtype: int64

In [122]:
data5['region'].value_counts()

region
Sierra                  7586
Pichincha               4165
Guayas                  3761
Costa, Santo Domingo    3753
Amazonía                3521
El Oro                  3315
Azuay                   2702
Manabí                   687
Los Ríos                 560
Galápagos                289
Name: count, dtype: int64

In [123]:
data6['region'].value_counts()

region
Sierra                  7593
Pichincha               4112
Guayas                  3955
Costa, Santo Domingo    3776
Amazonía                3610
El Oro                  3263
Azuay                   2714
Manabí                   708
Los Ríos                 545
Galápagos                322
Name: count, dtype: int64

In [124]:
data7['region'].value_counts()

region
Sierra                  7509
Pichincha               4065
Costa, Santo Domingo    3899
Guayas                  3823
Amazonía                3269
El Oro                  3268
Azuay                   2621
Manabí                   646
Los Ríos                 569
Galápagos                256
Name: count, dtype: int64

In [125]:
data8['region'].value_counts()

region
Sierra                  7524
Pichincha               4153
Costa, Santo Domingo    3798
Guayas                  3782
Amazonía                3420
El Oro                  3219
Azuay                   2631
Manabí                   691
Los Ríos                 569
Galápagos                305
Name: count, dtype: int64

In [126]:
data9['region'].value_counts()

region
Sierra                  7678
Pichincha               4209
Guayas                  3862
Costa, Santo Domingo    3759
Amazonía                3517
El Oro                  3189
Azuay                   2644
Manabí                   663
Los Ríos                 578
Galápagos                325
Name: count, dtype: int64

In [127]:
data10['region'].value_counts()

region
Sierra                  7768
Pichincha               4137
Guayas                  3943
Costa, Santo Domingo    3888
Amazonía                3612
El Oro                  3190
Azuay                   2679
Manabí                   682
Los Ríos                 589
Galápagos                326
Name: count, dtype: int64

In [128]:
data11['region'].value_counts()

region
Sierra                  7617
Pichincha               4082
Guayas                  3858
Costa, Santo Domingo    3763
Amazonía                3495
El Oro                  3263
Azuay                   2623
Manabí                   720
Los Ríos                 577
Galápagos                331
Name: count, dtype: int64

In [129]:
data12['region'].value_counts()

region
Sierra                  7535
Pichincha               4178
Costa, Santo Domingo    3822
Guayas                  3800
Amazonía                3423
El Oro                  3140
Azuay                   2607
Manabí                   694
Los Ríos                 520
Galápagos                307
Name: count, dtype: int64

## Calculo ingreso de los hogares

### Tenemos que crear un id de hogar sintético para los meses 1 al cinco

In [130]:
def build_household_id(df, use_p20=False, max_fallback_size=None, sort_cols=None):
    """
    Construye el id de hogar dentro de cada upm usando p04 (jefe de hogar), p01 (número de persona),
    más área y estrato.

    Params
    - df: DataFrame
    - use_p20: if True, usa p20 también, podría ser raro incluirlo
    - max_fallback_size: int or None. Si elegido, asigna un límite al número de personas en el hogar
    - sort_cols: opcional ordena columnas
    """
    df = df.copy()

    # Standardize types
    # p01 as string with left pad to 2 if looks numeric-ish
    def _pad_p01(x):
        if pd.isna(x):
            return None
        s = str(x).strip()
        # Drop decimal if looks like '01.0'
        if s.endswith('.0'):
            s = s[:-2]
        # keep non-numeric as-is
        if s.isdigit():
            return s.zfill(2)
        return s
    df['p01_std'] = df['p01'].map(_pad_p01)

    # p04 to numeric int where possible
    def _to_int_or_nan(x):
        try:
            v = int(float(x))
            return v
        except:
            return np.nan
    df['p04_int'] = df['p04'].map(_to_int_or_nan)

    # Key for grouping within UPM
    group_keys = ['upm', 'area', 'estrato']
    if use_p20 and 'p20' in df.columns:
        group_keys.append('p20')

    # Sorting within groups
    internal_sort = []
    # Put heads first
    internal_sort.append(df['p04_int'].fillna(0).eq(2).astype(int) * -1)  # heads first by negative flag
    # Then by p01 if numeric-like
    p01_num = pd.to_numeric(df['p01_std'], errors='coerce')
    internal_sort.append(p01_num.fillna(9999))
    # Optional stabilizers
    if sort_cols:
        for col in sort_cols:
            internal_sort.append(df[col])

    # Create a stable rank within each group for deterministic ordering
    df['_sort_key'] = pd.MultiIndex.from_arrays([s for s in internal_sort]).codes[0] if len(internal_sort)==1 else None
    # We'll actually sort with sort_values and groupby apply for clarity

    # Function to assign local household groups within a segment
    def assign_local_groups(seg: pd.DataFrame) -> pd.Series:
        s = seg.sort_values(
            by=['p04_int', 'p01_std'] + ([c for c in sort_cols] if sort_cols else []),
            ascending=[False, True] + ([True]*len(sort_cols) if sort_cols else [])
        ).copy()

        # Identify heads
        is_head = s['p04_int'].eq(2)
        if is_head.any():
            # Each head starts a new group; forward-fill to next head
            starters = is_head.astype(int)
            # cumulative sum across sorted rows to get group numbers starting at 1
            grp = starters.cumsum()
            # But rows before the first head will have grp=0; we can backfill them into the first group
            if (grp == 0).any():
                first_head_idx = np.argmax(is_head.values) if is_head.any() else None
                # If there are rows before first head, set them to 1
                grp = grp.mask(grp==0, 1)
            # Now handle multiple heads within what should be same household – by construction they get different grp IDs, which is desired
            local_grp = grp
        else:
            # Fallback: no heads in this segment.
            # Strategy A: start a new household whenever p01 == '01'
            p01_is_01 = s['p01_std'].eq('01')
            if p01_is_01.any():
                local_grp = p01_is_01.cumsum()
            else:
                # Strategy B: just bucket sequentially and optionally cap size
                n = len(s)
                if max_fallback_size and max_fallback_size > 0:
                    # group index = floor(pos / max_size) + 1
                    idx = np.arange(n)
                    local_grp = (idx // max_fallback_size) + 1
                else:
                    local_grp = np.ones(n, dtype=int)

        # Restore original order
        local_grp = pd.Series(local_grp.values, index=s.index)
        local_grp = local_grp.reindex(seg.index)
        # If there are NaNs (shouldn't), fill with 1
        return local_grp.fillna(1).astype(int)

    # Apply per segment
    df['local_hh'] = (
        df.groupby(group_keys, group_keys=False, sort=False)
        .apply(assign_local_groups)
    )

    # Build household_id as string
    # Make sure upm is string
    df['upm_str'] = df['upm'].astype(str).str.strip()
    df['area_str'] = df['area'].astype(str).str.strip()
    df['estrato_str'] = df['estrato'].astype(str).str.strip()
    df['idef_hogar'] = (
        df['upm_str'] + df['area_str'] + df['estrato_str'] + df['local_hh'].astype(str).str.zfill(3)
    )

    # Sanity flags
    hh_heads = df.groupby('idef_hogar', as_index=False)['p04_int'].apply(lambda x: np.nansum(x==2))
    hh_heads = hh_heads.rename(columns={'p04_int':'n_heads'})
    df = df.merge(hh_heads, on='idef_hogar', how='left')
    df['flag_multiple_heads'] = df['n_heads'] > 1
    df['flag_no_head'] = df['n_heads'] == 0

    # Cleanup
    df = df.drop(columns=['_sort_key'], errors='ignore')

    return df

In [131]:
data1 = build_household_id(data1)
data2 = build_household_id(data2)
data3 = build_household_id(data3)
data4 = build_household_id(data4)
data5 = build_household_id(data5)

/tmp/ipykernel_90761/2445721441.py:105: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_local_groups)
/tmp/ipykernel_90761/2445721441.py:105: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_local_groups)
/tmp/ipykernel_90761/2445721441.py:105: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping colu

In [132]:
for base in [data6, data7, data8, data9, data10, data11, data12]:
    base['idef_hogar'] = base['id_hogar']

for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    print(base['idef_hogar'].nunique())

5170
5060
5065
5239
5164
8713
8539
8613
8681
8791
8767
8736


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [133]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [134]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base['ingr_h'] = base.groupby('idef_hogar')['ingr_r'].transform(sum_with_na)

In [135]:
for base in [data1, data2, data3, data4, data5]:
    base['ingr_h'] = base['ingr_h'] / 2

In [136]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    print(base['ingr_h'].mean())

851.1290740471043
568.1388640825573
544.5889304774154
601.687249641048
555.0961975172177
644.916052782176
635.8267879109526
632.3401776126191
740.929893904461
646.7443115842538
636.4680171624622
661.7465276104701


## Sacamos edades negativas y mayores a 100 años

Transformamos las variables de edad a numericas para evitar problemas

In [137]:
for base in [data1, data2, data3, data4, data5, data6, data7, 
             data8, data9, data10, data11, data12]:
    base['edad'] = pd.to_numeric(base['p03'], errors='coerce')

In [138]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base = base.loc[(base['edad'] >= 0) & (base['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [139]:
k = 0.4
s = 0.9

In [140]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    # Si es necesario calcular el número de niños
    base['es_nino'] = base['edad'] < 10 
    base['ninos'] = base.groupby('idef_hogar')['es_nino'].transform('sum')

    # Si es necesario calcular el número de adultos
    base['es_adulto'] = base['edad'] > 10
    base['adultos'] = base.groupby('idef_hogar')['es_adulto'].transform('sum')

In [141]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base['escala'] = (base['adultos'] + k * base['ninos']) ** s

In [142]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
    base['ingr_t'] = base['ingr_h'] / base['escala']

In [143]:
for base in [data1, data2, data3, data4, data5]:
    base['ingr_t'] = base['ingr_t'] * 2

In [144]:
for base in [data1, data2, data3, data4, data5, data6, data7, data8, data9,
             data10, data11, data12]:
        print(base['ingr_t'].mean())

230.0282043749324
149.3526200620843
145.85034027469297
155.8256062692796
144.20479570190682
200.34532443800074
193.04114807410213
193.5077774836448
229.599917597855
198.00191824780677
195.9645363255348
205.8636067059461


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [145]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2021

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [146]:
for base in [data10, data11, data12]:
    base.columns = base.columns.str.lower()

In [147]:
resultados_list = []

# Diccionario de dataframes mensuales
data_dict = {
    1: data1, 2: data2, 3: data3, 4: data4, 
    5: data5, 6: data6, 7: data7, 8: data8,
    9: data9, 10: data10, 11: data11, 12: data12
}

# Para cada mes
for mes in range(1, 13):
    col_ingr = f'ingr_t'
    
    # Calcula trimestre (1-4) basado en el mes
    trimestre = (mes - 1) // 3 + 1
    
    umbral = umbral_dict.get(trimestre)
    salario = salario_dict.get(trimestre)
    df_actual = data_dict[mes]
    
    # Agrupa por región
    grouped = df_actual.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'mes': mes,
            'trimestre': trimestre,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,mes,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2021,1,1,Amazonía,0.491500,0.276522,0.191361,0.164953,0.325232,0.503559,183.504879,81.553106,400.0,4.904779
1,2021,1,1,Azuay,0.332056,0.143115,0.077341,0.113055,0.214869,0.308571,224.048159,159.427927,400.0,2.508971
2,2021,1,1,"Costa, Santo Domingo",0.445505,0.232715,0.150831,0.124810,0.236012,0.335262,140.883736,86.721637,400.0,4.612459
3,2021,1,1,El Oro,0.300085,0.109281,0.059108,0.099086,0.194431,0.295262,207.670993,123.048735,400.0,3.250745
4,2021,1,1,Galápagos,0.167099,0.112099,0.081376,0.103609,0.216925,0.349516,517.832177,333.397735,400.0,1.199768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2021,12,4,Guayas,0.252274,0.091660,0.049903,0.062526,0.123898,0.192895,159.485182,134.833442,400.0,2.966623
116,2021,12,4,Los Ríos,0.351858,0.132456,0.084324,0.091312,0.177190,0.258124,155.019368,94.094544,400.0,4.251044
117,2021,12,4,Manabí,0.549627,0.286270,0.180289,0.104989,0.211904,0.344000,108.003083,66.678466,400.0,5.998938
118,2021,12,4,Pichincha,0.122253,0.048162,0.032985,0.110460,0.207401,0.303850,278.502899,169.250076,400.0,2.363367


In [148]:
# Agregar por año, trimestre y región
df_trimestral = df_final_regional.groupby(['ano', 'trimestre', 'region']).agg({
    'fgt0': 'mean',
    'fgt1': 'mean',
    'fgt2': 'mean',
    'a25': 'mean',
    'a50': 'mean',
    'a75': 'mean',
    'ingreso_promedio': 'mean',
    'ingreso_mediana': 'mean',
    'salario_minimo': 'mean',
    'kaitz_indice': 'mean'
}).reset_index()

df_trimestral

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2021,1,Amazonía,0.493758,0.256425,0.168995,0.121490,0.239562,0.366457,139.163113,80.352718,400.0,4.979306
1,2021,1,Azuay,0.358521,0.138555,0.073371,0.090764,0.174363,0.256426,171.984189,126.287768,400.0,3.267280
2,2021,1,"Costa, Santo Domingo",0.470749,0.236732,0.151093,0.096451,0.189660,0.287428,117.391126,82.975524,400.0,4.826540
3,2021,1,El Oro,0.339192,0.145004,0.085198,0.086670,0.170756,0.257748,164.292685,117.386552,400.0,3.411739
4,2021,1,Galápagos,0.219822,0.104776,0.062398,0.103873,0.203070,0.301712,321.574622,192.693396,400.0,2.588265
5,2021,1,Guayas,0.360177,0.148805,0.090137,0.086022,0.170285,0.260292,162.486199,120.009190,400.0,3.457654
6,2021,1,Los Ríos,0.424750,0.207034,0.127122,0.080636,0.162499,0.254554,129.925378,100.438506,400.0,4.383926
7,2021,1,Manabí,0.476815,0.193760,0.116974,0.087763,0.175323,0.275165,125.549205,87.137374,400.0,4.715738
8,2021,1,Pichincha,0.267524,0.112126,0.065494,0.108923,0.209469,0.311295,238.419232,155.672863,400.0,2.636736
9,2021,1,Sierra,0.412741,0.196289,0.125469,0.102063,0.199778,0.298965,146.110919,95.711750,400.0,4.264647


### Inserta los cálculos en la base final

In [149]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [150]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_trimestral.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_trimestral.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')